In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from cuml.decomposition import PCA
from cuml.manifold import UMAP


# fill these
RUN_NAME = ...
PATH_EMBEDDINGS = Path(...)
PATH_TRACK_LOOKUP = Path(...)
PATH_OUTPUT_DIR = Path(...)

if not isinstance(RUN_NAME, str):
    raise ValueError("Set RUN_NAME to the run tag string, e.g. 'vivid_dragon'")
if not PATH_EMBEDDINGS.exists():
    raise FileNotFoundError(f"Could not find `{PATH_EMBEDDINGS}`")
print(f"Set embedding to `{PATH_EMBEDDINGS}`")
if not PATH_TRACK_LOOKUP.exists():
    raise FileNotFoundError(f"Could not find `{PATH_TRACK_LOOKUP}`")
print(f"Set track lookup to `{PATH_TRACK_LOOKUP}`")
PATH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Set output directory to `{PATH_OUTPUT_DIR}`.")

In [ ]:
print("Loading embeddings ...")
embs_df = pd.read_parquet(PATH_EMBEDDINGS)
emb_cols = [c for c in embs_df.columns if c.startswith("e") and c[1:].isdigit()]
print(f"{len(embs_df):,} tracks × {len(emb_cols)} dims")

print("Loading lookup ...")
lookup_df = pd.read_parquet(PATH_TRACK_LOOKUP)
print(f"{len(lookup_df):,} tracks × {len(lookup_df.columns)} dims")

embs_meta = embs_df[["track_rowid"]].merge(
    lookup_df[["track_rowid", "logcounts", "track_name", "artist_name"]],
    on="track_rowid",
    how="left",
)

# Track embeddings

In [ ]:
nc = 2
nn = 140
md = 0.025
local_connectivity = 1.0
spread = 0.50
metric = "cosine"
init = "spectral"
random_state = None

# PCA pre-projection: reduce to PCA_DIMS before UMAP.
# Set to None to skip PCA.
PCA_DIMS = None

# Top-N tracks by logcount for UMAP fit. None = use full dataset.
N_SAMPLE = 1_000_000

# True  → fit on N_SAMPLE, transform full dataset (production, slow).
# False → fit_transform on N_SAMPLE only (exploration, fast).
TRANSFORM_ALL = False

In [ ]:
matrix = embs_df[emb_cols].to_numpy(dtype=np.float32)

if PCA_DIMS is not None:
    print(f"PCA {matrix.shape[1]}d → {PCA_DIMS}d ...")
    pca = PCA(n_components=PCA_DIMS, random_state=0)
    matrix = pca.fit_transform(matrix)
    print(f"  explained variance ratio sum: {pca.explained_variance_ratio_.sum():.3f}")
else:
    pca = None

In [ ]:
reducer = UMAP(
    n_components=nc,
    n_neighbors=nn,
    min_dist=md,
    local_connectivity=local_connectivity,
    spread=spread,
    metric=metric,
    init=init,
    random_state=random_state,
    verbose=False,
)

if N_SAMPLE is not None and N_SAMPLE < len(embs_df):
    sample_idx = embs_meta["logcounts"].nlargest(N_SAMPLE).index.to_numpy()
    sample_matrix = matrix[sample_idx]
    print(f"Sample: {len(sample_idx):,} / {len(matrix):,} tracks (top by logcount)")
else:
    sample_idx = None
    sample_matrix = matrix
    print(f"Using full dataset: {len(matrix):,} tracks")

if TRANSFORM_ALL:
    print("Fitting on sample (transform step is in the next cell) ...")
    reducer.fit(sample_matrix)
    print("Fit done.")
else:
    print("fit_transform on sample ...")
    coords = reducer.fit_transform(sample_matrix)
    export_rowids = embs_df["track_rowid"].values[sample_idx] if sample_idx is not None else embs_df["track_rowid"].values
    print(f"Done. ({len(coords):,} rows)")

In [ ]:
# Re-run this cell alone (with TRANSFORM_ALL=True) to project the full dataset
# onto a previously fitted reducer, without re-fitting.
if TRANSFORM_ALL:
    print(f"Transforming {len(matrix):,} tracks ...")
    coords = reducer.transform(matrix)
    export_rowids = embs_df["track_rowid"].values
    print(f"Done. ({len(coords):,} rows)")
else:
    print("Exploration mode — full transform skipped.")

In [ ]:
md_str = str(md).replace(".", "d")

coord_cols = {
    f"umap_{('xyz'[j] if j < 3 else str(j))}": coords[:, j].astype(np.float32)
    for j in range(nc)
}
track_df = pd.DataFrame({"track_rowid": export_rowids, **coord_cols})

### Export

In [ ]:
if sample_idx is None or TRANSFORM_ALL:
    out_path = PATH_OUTPUT_DIR / f"{RUN_NAME}_umap_track_{nc}d_nn{nn}_md{md_str}_{metric}.parquet"
    track_df.to_parquet(out_path, index=False)
    print(f"  → {out_path}  ({len(track_df):,} rows)")
else:
    print(f"Exploration mode — skipping export.  ({len(track_df):,} rows)")

# Artists, albums, labels

In [ ]:
from src.entities import Albums


album_df = Albums.embeddings(embs_df, lookup_df)
album_matrix = album_df[emb_cols].to_numpy(dtype=np.float32)
if pca is not None:
    album_matrix = pca.transform(album_matrix)
coords = reducer.transform(album_matrix)

out_path = PATH_OUTPUT_DIR / f"{RUN_NAME}_umap_album_{nc}d_nn{nn}_md{md_str}_{metric}.parquet"
coord_cols = {
    f"umap_{('xyz'[j] if j < 3 else str(j))}": coords[:, j].astype(np.float32)
    for j in range(nc)
}
df = pd.DataFrame(
    {
        "album_rowid": album_df["album_rowid"].values,
        **coord_cols
    }
)
df.to_parquet(out_path, index=False)
print(f"  → {out_path}  ({len(coords):,} rows)")

In [ ]:
from src.entities import Artists


artist_df = Artists.embeddings(embs_df, lookup_df)
artist_matrix = artist_df[emb_cols].to_numpy(dtype=np.float32)
if pca is not None:
    artist_matrix = pca.transform(artist_matrix)
coords = reducer.transform(artist_matrix)

out_path = PATH_OUTPUT_DIR / f"{RUN_NAME}_umap_artist_{nc}d_nn{nn}_md{md_str}_{metric}.parquet"
coord_cols = {
    f"umap_{('xyz'[j] if j < 3 else str(j))}": coords[:, j].astype(np.float32)
    for j in range(nc)
}
df = pd.DataFrame(
    {
        "artist_rowid": artist_df["artist_rowid"].values,
        **coord_cols
    }
)
df.to_parquet(out_path, index=False)
print(f"  → {out_path}  ({len(coords):,} rows)")

In [ ]:
from src.entities import Labels


label_df = Labels.embeddings(embs_df, lookup_df)
label_matrix = label_df[emb_cols].to_numpy(dtype=np.float32)
if pca is not None:
    label_matrix = pca.transform(label_matrix)
coords = reducer.transform(label_matrix)

out_path = PATH_OUTPUT_DIR / f"{RUN_NAME}_umap_label_{nc}d_nn{nn}_md{md_str}_{metric}.parquet"
coord_cols = {
    f"umap_{('xyz'[j] if j < 3 else str(j))}": coords[:, j].astype(np.float32)
    for j in range(nc)
}
df = pd.DataFrame(
    {
        "label": label_df["label"].values,
        **coord_cols
    }
)
df.to_parquet(out_path, index=False)
print(f"  → {out_path}  ({len(coords):,} rows)")

# Visualize

In [ ]:
BINS = 1000

x = track_df["umap_x"].to_numpy()
y = track_df["umap_y"].to_numpy()

x_min, x_max = x.min(), x.max()
y_min, y_max = y.min(), y.max()

cx, cy = (x_min + x_max) / 2, (y_min + y_max) / 2
print(f"x  [{x_min:.3f}, {x_max:.3f}]  range {x_max - x_min:.3f}  center {cx:.3f}  std {x.std():.3f}")
print(f"y  [{y_min:.3f}, {y_max:.3f}]  range {y_max - y_min:.3f}  center {cy:.3f}  std {y.std():.3f}")
print(f"{len(x):,} points")

xi = np.clip(((x - x_min) / (x_max - x_min) * BINS).astype(np.int32), 0, BINS - 1)
yi = np.clip(((y - y_min) / (y_max - y_min) * BINS).astype(np.int32), 0, BINS - 1)

heatmap = np.bincount(yi * BINS + xi, minlength=BINS * BINS).reshape(BINS, BINS)

viz = track_df.merge(
    embs_meta[["track_rowid", "artist_name", "track_name", "logcounts"]],
    on="track_rowid",
    how="left",
)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm


def plot_umap(highlights=None):
    """Plot the UMAP heatmap with optional track overlays.

    highlights: list of (artist, track_name) tuples (case-insensitive).
                Unmatched entries are printed as warnings.
    """
    with plt.style.context("dark_background"):
        fig, ax = plt.subplots(figsize=(10, 10))
        im = ax.imshow(
            heatmap,
            origin="lower",
            norm=PowerNorm(gamma=0.3, vmin=0., vmax=heatmap.max()),
            cmap="inferno",
            interpolation="nearest",
            extent=[x_min, x_max, y_min, y_max],
            aspect="auto",
        )
        fig.colorbar(im, ax=ax, fraction=0.046, shrink=0.3, pad=0.04)
        ax.set_axis_off()

        if highlights:
            for artist, track in highlights:
                matches = viz[
                    (viz["artist_name"].str.lower() == artist.lower()) &
                    (viz["track_name"].str.lower() == track.lower())
                ]
                if matches.empty:
                    print(f"  not found: {artist!r} – {track!r}")
                    continue
                row = matches.sort_values("logcounts", ascending=False).iloc[0]
                ax.scatter(row["umap_x"], row["umap_y"], s=80, zorder=5,
                           edgecolors="white", linewidths=0.8)
                ax.annotate(f"{artist} – {track}", (row["umap_x"], row["umap_y"]),
                            fontsize=7, color="white",
                            xytext=(6, 4), textcoords="offset points")

        plt.tight_layout()
        plt.show()


plot_umap([
    # ("Radiohead", "Karma Police"),
])